# Colab: real-data PINN, batch 2 (fixes from batch 1)

**What batch 1 showed.**
- Storage S drifted to almost zero in 3 of the 9 runs (as low as 0.0002), so the time part of the physics switched off and the 2019 forecasts blew up.
- At places with no well, the model ignored the land surface.

**What this batch tests.**
- S is kept inside the CGWB report range (0.072–0.15).
- The model works on *water level minus smoothed ground height*, so between wells the water table follows the land.

It runs one regional model and the 4 zone models with these fixes, plus 3 regional variants. Results are scored on the **validation** wells only.

**Before you start:**
1. Upload the new `pinn_code.zip` (from the laptop's `pinn` folder) to `MyDrive/pinn/`, replacing the old one.
2. Choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

**Timing.** About 2.7 hours (8 runs of about 20 minutes). Keep the tab open.

**If Colab disconnects:** just do **Run all** again. Runs that already finished are skipped.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/pinn"

In [ ]:
import glob
import os
import shutil
import zipfile
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/runs/logs", exist_ok=True)
CODE = "/content/pinn"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_ROOT}/pinn_code.zip") as z:
    z.extractall(CODE)
%cd {CODE}

In [ ]:
# PyTorch is already installed on Colab; this adds everything else (about 1 minute).
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ["PINN_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"
!python scripts/check_env.py
!python -m pytest -q -m "not data"

## Runs (most important first)

| Run | Question it answers |
|---|---|
| regional `tb` (terrain + S in report range) | the main comparison model with both fixes |
| zones 1–4 `tb` | our method, same settings |
| regional `tbw` (S range widened to 0.02–0.25) | does S still want to go low when it may? |
| regional `sr` (S range, no terrain) | how much does the terrain fix help on its own? |
| regional `tb_off1` (weaker well corrections) | batch 1's best forecast setting, now with the fixes |

**How the setting is chosen (fixed before the runs, same as batch 1):** lowest combined validation error on tasks A and B, each relative to the best baseline for that task.

In [ ]:
def done(pattern):
    return bool(glob.glob(f"{DRIVE_ROOT}/runs/{pattern}/metrics.json"))

LOG = f"{DRIVE_ROOT}/runs/logs/real2.log"
RUNS = [
    ("*_real_L0_regional_s0_tb", "--regional --tag tb"),
    *[(f"*_real_L0_kmeansk5_z{z}_s0_tb", f"--zone {z} --tag tb") for z in (1, 2, 3, 4)],
    ("*_real_L0_regional_s0_tbw", "--regional --tag tbw --set training.S_range=[0.02,0.25]"),
    ("*_real_L0_regional_s0_sr", "--regional --tag sr --set h_net.terrain_sigma_km=0"),
    ("*_real_L0_regional_s0_tb_off1", "--regional --tag tb_off1 --set training.well_offsets.weight=1"),
]
for pattern, args in RUNS:
    if done(pattern):
        print("already done:", pattern)
        continue
    !python -u scripts/run_real.py {args} --stage-only val 2>&1 | tee -a {LOG}

## Bundle the results for Claude
This makes `MyDrive/pinn/real2_results.zip`. Download it, put it in the `pinn` folder on the laptop, then tell Claude.

In [ ]:
out = f"{DRIVE_ROOT}/real2_results.zip"
TAGS = ("_s0_tb", "_s0_tbw", "_s0_sr", "_s0_tb_off1")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob(f"{DRIVE_ROOT}/runs/**/*", recursive=True):
        rel = os.path.relpath(path, f"{DRIVE_ROOT}/runs")
        run = rel.split(os.sep)[0]
        if os.path.isfile(path) and (("_real_" in run and run.endswith(TAGS)) or rel.endswith("real2.log")):
            z.write(path, rel)
print("wrote", out, f"({os.path.getsize(out) / 1e6:.2f} MB)")